# 🤖 Stage 1 — Dataset Generation for RAG

## Project Overview

This notebook implements the first stage of the project: the automatic generation of an instruction-response dataset from a domain-specific document.

The knowledge source used in this work is the official user manual of the **Midea MFM01D110WB 11 kg Washer-Dryer**. The generated dataset will be used in subsequent stages for LoRA-based fine-tuning, model evaluation, and RESTful API integration as part of a complete Retrieval-Augmented Generation (RAG) system.

---

## 📌 Objectives

* Extract textual content from the PDF manual.
* Split the document into smaller text chunks.
* Generate instruction-response pairs using a Large Language Model (LLM).
* Perform manual curation of the generated examples.
* Export the final dataset in JSONL format for fine-tuning purposes.

---

## 📖 Knowledge Source

**Document:** Midea MFM01D110WB 11 kg Washer-Dryer User Manual.

The manual contains information related to installation, operation, maintenance, safety recommendations, troubleshooting procedures, and technical specifications of the appliance.

This document was selected due to its structured technical content, objective instructions, and specialized knowledge, making it a suitable source for generating question-answer pairs for supervised language model training.

Furthermore, the manual represents a specific knowledge domain, allowing the evaluation of a model's ability to learn and reproduce technical information after the fine-tuning process.

---

## 💻 Dataset Generation Model

The instruction-response pairs will be generated using the following model:

* **Model:** SmolLM2-1.7B-Instruct
* **Number of Parameters:** 1.7 Billion
* **Task:** Instruction-Response Pair Generation

The model is used exclusively for the automatic construction of the supervised dataset and does not correspond to the models that will be fine-tuned and evaluated in the subsequent stages of the project.

---

## 📊 Expected Dataset Structure

The final dataset must follow the JSONL format, containing one example per line:

```json
{
    "Instruction": "Question",
    "Output": "Answer"
}
```

After the automatic generation process, all examples will undergo a manual curation stage to remove inconsistencies, correct errors, and eliminate possible hallucinations, ensuring the quality of the dataset used for fine-tuning.

---

## 🔄 Stage 1 Workflow

The dataset generation process will follow the steps below:

1. Extract text from the PDF document.
2. Split the content into chunks.
3. Filter relevant text segments.
4. Generate instruction-response pairs automatically.
5. Perform manual curation of the generated examples.
6. Export the final dataset in JSONL format.


# 📦 1. Dependency Installation

This project relies on a set of Python libraries for PDF processing, data manipulation, large language model inference, and dataset generation.

To ensure reproducibility and simplify environment setup, all required dependencies are specified in a `requirements.txt` file.

The main libraries used in this notebook include:

* **PyTorch** for model execution.
* **Transformers** for loading and running the language model.
* **Accelerate** for efficient model inference.
* **PDFPlumber** for extracting text from PDF documents.
* **Pandas** for data manipulation and analysis.
* **TQDM** for progress monitoring during dataset generation.

Install the required packages by executing the following command:


In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# 📚 2. Importing Libraries

This section imports the libraries required for the dataset generation pipeline.

The imported packages provide functionality for PDF text extraction, data processing, large language model inference, progress monitoring, and file manipulation.

After importing the libraries, the execution environment will be verified to ensure that all dependencies are correctly installed and available.


In [2]:
# File handling
import json

import re

import fitz

# Data processing
import pandas as pd

# PDF text extraction
import pdfplumber

# Progress monitoring
from tqdm import tqdm

# Deep Learning
import torch

# Hugging Face Transformers
from transformers import pipeline

In [3]:
print("Torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

Torch: 2.12.0+cpu
CUDA disponível: False


# 📝 3. Extracting Text from the Knowledge Source

This section defines the function responsible for extracting textual content from the PDF document that serves as the project's knowledge base.


In [4]:
def extract_text_from_pdf_blocks(pdf_path):
    """
    Extracts text from a PDF using text blocks.

    This approach works better for technical manuals
    containing multiple columns, warning boxes and
    visual layouts.

    Parameters
    ----------
    pdf_path : str

    Returns
    -------
    str
    """

    doc = fitz.open(pdf_path)

    blocks_text = []

    for page in doc:

        blocks = page.get_text("blocks")

        # Ordena os blocos pela posição na página
        blocks = sorted(
            blocks,
            key=lambda b: (b[1], b[0])
        )

        for block in blocks:

            text = block[4].strip()

            if not text:
                continue

            blocks_text.append(text)

    doc.close()

    return "\n\n".join(blocks_text)

PDF_PATH = "../data/pdf/manual.pdf"

manual_text = extract_text_from_pdf_blocks(
    PDF_PATH
)

print(
    f"Total characters extracted: "
    f"{len(manual_text):,}"
)

print("\nPreview:\n")

print(manual_text[:5000])

Total characters extracted: 83,236

Preview:

LAVA E SECA

MANUAL DO USUÁRIO

11kg

MODELOS:
MFM01D110WB

www.midea.com/br

Obrigado por escolher a Midea!

A Midea é uma empresa comprometida com o bem-estar das pessoas. Com a 
combinação de design inteligente e tecnologia, seu novo equipamento trará 
novas experiências e deixará seu dia a dia muito mais agradável. Uma receita 
simples que fez da Midea uma das maiores fabricantes de eletrodomésticos e 
condicionadores de ar do mundo.

Este manual foi feito especialmente para que você conheça todas as características 
do seu aparelho, além de informações sobre manutenção, execução de serviços 
e claro, como obter o máximo das suas funcionalidades.

Caso precise de informações adicionais ou tenha dúvidas sobre a garantia, 
entre em contato através do nosso Serviço de Atendimento ao Consumidor, 
pelos telefones ou pelo site abaixo.

SAC - Serviço de Atendimento ao Consumidor

3003 1005 (capitais e regiões metropolitanas)

0800 648 1005 (de

## Saving Extracted Text


In [73]:
with open(
    "../data/processed/manual_extracted.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(manual_text)

print("Extracted text saved successfully.")

Extracted text saved successfully.


# ☑️ 4. Text Chunking Strategy

Large Language Models cannot efficiently process an entire document at once due to context length limitations. Therefore, the extracted text must be divided into smaller segments, commonly referred to as chunks.

The chunking process directly impacts the quality of the generated instruction-response pairs. Smaller chunks tend to produce more focused questions and answers, while larger chunks provide additional context but may introduce irrelevant information.

---

To evaluate this trade-off, two chunk sizes will be tested:

- 300-character chunks;
- 500-character chunks;
- 1000-character chunks;

TAll generated chunks were stored as JSONL files to ensure traceability, reproducibility, and support further analysis during dataset creation.


## 4. 1 Remove Table of Contents

To guarantee efficient and relevant chunks, removing the table of contents is crucial for a good quality chunk.

In [5]:
def remove_table_of_contents(text):

    pattern = (
        r'CONTEÚDO.*?'
        r'OBSERVAÇÕES IMPORTANTES\s+'
        r'O manual do usuário'
    )

    replacement = (
        'OBSERVAÇÕES IMPORTANTES\n'
        'O manual do usuário'
    )

    cleaned_text = re.sub(
        pattern,
        replacement,
        text,
        flags=re.DOTALL
    )

    return cleaned_text

## 4. 2 Remove Headers and Footers

In [6]:
def clean_manual_text(text):

    text = re.sub(
        r'\d+\s*MFM01D110WB_USER MANUAL\s*\([^)]*\)',
        '',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    return text.strip()

In [7]:
manual_text = extract_text_from_pdf_blocks(
    PDF_PATH
)

manual_text = remove_table_of_contents(
    manual_text
    
)
manual_text = clean_manual_text(
    manual_text
)

## 4. 3 Split Chunks Function

In [27]:
def split_text(text, max_chunk_length=500):

    text = " ".join(text.split())

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    chunks = []
    current_chunk = ""

    for sentence in sentences:

        if len(current_chunk) + len(sentence) + 1 <= max_chunk_length:

            current_chunk += sentence + " "

        else:

            chunks.append(current_chunk.strip())

            current_chunk = sentence + " "

    if current_chunk:

        chunks.append(current_chunk.strip())

    return chunks

## 4. 3. 1 Chunk Size Comparison

To analyze the impact of chunk size on dataset generation, two configurations will be tested:

- 500 characters
- 1000 characters

The resulting chunks will be compared in terms of quantity and content coverage.

In [46]:
chunks_300 = split_text(
    manual_text,
    max_chunk_length=300
)

chunks_500 = split_text(
    manual_text,
    max_chunk_length=500
)

chunks_1000 = split_text(
    manual_text,
    max_chunk_length=1000
)


print(f"300-character chunks: {len(chunks_300)}")
print(f"500-character chunks: {len(chunks_500)}")
print(f"1000-character chunks: {len(chunks_1000)}")

300-character chunks: 302
500-character chunks: 172
1000-character chunks: 81


## 4. 3. 2 Sample Chunks

The following cells display examples of generated chunks for manual inspection.

### Testing Chunk Size: 300


In [29]:
print(chunks_300[0])

LAVA E SECA MANUAL DO USUÁRIO 11kg MODELOS: MFM01D110WB www.midea.com/br Obrigado por escolher a Midea! A Midea é uma empresa comprometida com o bem-estar das pessoas.


### Testing Chunk Size: 500


In [30]:
print(chunks_500[0])

LAVA E SECA MANUAL DO USUÁRIO 11kg MODELOS: MFM01D110WB www.midea.com/br Obrigado por escolher a Midea! A Midea é uma empresa comprometida com o bem-estar das pessoas. Com a combinação de design inteligente e tecnologia, seu novo equipamento trará novas experiências e deixará seu dia a dia muito mais agradável. Uma receita simples que fez da Midea uma das maiores fabricantes de eletrodomésticos e condicionadores de ar do mundo.


### Testing Chunk Size: 1000


In [11]:
print(chunks_1000[0])

LAVA E SECA MANUAL DO USUÁRIO 11kg MODELOS: MFM01D110WB www.midea.com/br Obrigado por escolher a Midea! A Midea é uma empresa comprometida com o bem-estar das pessoas. Com a combinação de design inteligente e tecnologia, seu novo equipamento trará novas experiências e deixará seu dia a dia muito mais agradável. Uma receita simples que fez da Midea uma das maiores fabricantes de eletrodomésticos e condicionadores de ar do mundo. Este manual foi feito especialmente para que você conheça todas as características do seu aparelho, além de informações sobre manutenção, execução de serviços e claro, como obter o máximo das suas funcionalidades. Caso precise de informações adicionais ou tenha dúvidas sobre a garantia, entre em contato através do nosso Serviço de Atendimento ao Consumidor, pelos telefones ou pelo site abaixo.


## 4. 3. 3 Save Chunk Sizes

In [49]:
import json
from pathlib import Path

def save_chunks(chunks, output_file):

    Path(output_file).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:

        for idx, chunk in enumerate(chunks):

            record = {
                "chunk_id": idx,
                "length": len(chunk),
                "chunk": chunk
            }

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

save_chunks(
    chunks_300,
    "../data/chunks/chunks_300.jsonl"
)

save_chunks(
    chunks_500,
    "../data/chunks/chunks_500.jsonl"
)

save_chunks(
    chunks_1000,
    "../data/chunks/chunks_1000.jsonl"
)

## 4. 3. 4 Saving Chunk Statistics

In [50]:
stats = pd.DataFrame({
    "strategy": [
        "300_chars",
        "500_chars",
        "1000_chars",
    ],
    "num_chunks": [
        len(chunks_300),
        len(chunks_500),
        len(chunks_1000),
    ]
})

stats

stats.to_csv(
    "../reports/chunk_statistics.csv",
    index=False
)

## 👾 5. Loading the Language Model

This section loads the Large Language Model (LLM) responsible for generating instruction-response pairs from the document chunks.

The selected model is **SmolLM2-1.7B-Instruct**, an instruction-tuned language model developed for efficient text generation and instruction-following tasks. With approximately **1.7 billion parameters**, the model satisfies the project requirement of using a local LLM with more than 1.5 billion parameters.

In [12]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading model: {MODEL_ID}")

Loading model: Qwen/Qwen2.5-3B-Instruct


In [13]:
hf_pipeline = pipeline(
    task="text-generation",
    model=MODEL_ID,
    device_map="cpu"
)

print("Model loaded successfully.")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded successfully.


# 👥 6. Generating Instruction-Response Pairs

This section generates instruction-response pairs from the document chunks using the selected language model.

Each generated pair consists of:

- An instruction (question).
- A response (answer).

Both elements must be derived exclusively from the information contained in the source chunk.


## 6.1 Chunk Quality Filter

Some chunks may contain metadata such as website URLs, phone numbers, copyright notices, or table of contents entries. These chunks are not useful for instruction tuning and may generate low-quality instruction-response pairs.

To improve dataset quality, a filtering function is applied before generation.


In [101]:
def is_relevant_chunk(chunk):

    chunk_lower = chunk.lower()

    if len(chunk.split()) < 15:
        return False

    unwanted_patterns = [
        "www.",
        "http",
        "0800",
        "atendimento ao consumidor",
        "logística reversa",
        "abree",
        "reciclagem",
        "manual do usuário",
        "esclarecer quaisquer dúvidas",
        "garantir o bom funcionamento",
        "indica ao usuário",
        "procedimentos que requerem maior atenção",
        "recomendações e advertências",
        "para garantir o melhor desempenho",
        "é de inteira responsabilidade do usuário"
    ]

    if any(p in chunk_lower for p in unwanted_patterns):
        return False

    operational_patterns = [
        "medidas importantes de segurança",
        "não deve",
        "nunca",
        "evite",
        "recomenda-se",
        "deve ser",
        "não utilize",
        "não instale",
        "não conecte",
        "limpeza",
        "manutenção",
        "instalação",
        "filtro",
        "porta",
        "plugue",
        "tomada",
        "cabo",
        "mangueira"
    ]

    return any(
        p in chunk_lower
        for p in operational_patterns
    )

## 6. 2 Applying Filter

In [102]:
relevant_chunks = [
    chunk
    for chunk in chunks_300
    if is_relevant_chunk(chunk)
]

print(f"Total chunks: {len(chunks_300)}")
print(f"Relevant chunks: {len(relevant_chunks)}")

Total chunks: 302
Relevant chunks: 155


## 6. 3 Inspection

In [40]:
for i in range(10):
    print("=" * 80)
    print(relevant_chunks[i])
    print()

Este manual foi feito especialmente para que você conheça todas as características do seu aparelho, além de informações sobre manutenção, execução de serviços e claro, como obter o máximo das suas funcionalidades.

1.1 - Medidas Importantes de Segurança NOTA Para reduzir os riscos de choques elétricos, queimaduras, ferimentos pessoais ou danos ao equipamento, siga as recomendações básicas de segurança ao usar este aparelho: • Este aparelho não deve ser instalado embutido.

• Evite instalar o aparelho em um local onde exista a incidência de raios solares.

• O aparelho não deve ser instalado atrás de uma porta com fechadura, uma porta de correr ou de uma porta com dobradiças no lado oposto à secadora, de um modo que a completa abertura da porta da lava e seca seja impedida PERIGO • Este aparelho não se destina à utilização por pessoas (inclusive crianças) com capacidades físicas, sensoriais ou mentais reduzidas, ou por pessoas com falta de experiência e conhecimento, a menos que tenham 

In [62]:
from transformers.utils import logging

logging.set_verbosity_error()

## 6. 2 Instruction-Response Generation

The following function prompts the language model to generate a practical user question and its corresponding answer based exclusively on the content contained in a document chunk.

The prompt is designed to prioritize operational, maintenance, safety, and troubleshooting information related to the washing machine.


In [90]:
def generate_instruction_response(chunk, hf_pipeline):

    prompt = f"""
You are generating a supervised fine-tuning dataset from a washing machine user manual.

Using only the content below, generate exactly one question-answer pair.

Rules:

* Use only information explicitly stated in the content.
* Do not infer, invent, or use external knowledge.
* Write both the question and answer in English only.
* The question must be specific and under 15 words.
* The answer must be direct and under 15 words.
* Focus on one fact, warning, instruction, recommendation, or error code.
* Avoid vague or generic questions.
* The question must identify the subject explicitly. Do not use vague formulations such as 'What should be done?', 'What is important?', or 'What should be included?' unless the subject is clearly specified.
* If no clear pair can be generated, output:

INSTRUCTION: INVALID
RESPONSE: INVALID

Format:

INSTRUCTION: <short question>
RESPONSE: <short answer>

Content:
{chunk}
"""
    
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    try:

        outputs = hf_pipeline(
            messages,
            max_new_tokens=80,
            return_full_text=False,
            do_sample=False
        )

        generated_text = outputs[0]["generated_text"].strip()

        if "INSTRUCTION:" not in generated_text:
            return None, None

        if "RESPONSE:" not in generated_text:
            return None, None

        instruction = (
            generated_text
            .split("INSTRUCTION:")[1]
            .split("RESPONSE:")[0]
            .strip()
        )

        response = (
            generated_text
            .split("RESPONSE:")[1]
            .strip()
        )

        if not instruction or not response:
            return None, None

        return instruction, response

    except Exception as e:

        print(f"Generation error: {e}")

        return None, None

### 6.3 Single Example Test

Before generating the full dataset, a single chunk is tested to verify that the model produces instruction-response pairs in the expected format.


In [104]:
print(chunks_300[18])
print(chunks_300[19])
print(chunks_300[20])
print(chunks_300[21])
print(chunks_300[22])

A adaptação e a preparação do local para a instalação do produto, tais como: rede hidráulica, alvenaria, carpintaria, preparação da rede elétrica do ambiente (tomada, disjuntor, bitola de cabos, eletroduto, etc), é de inteira responsabilidade do usuário/consumidor.
1.1 - Medidas Importantes de Segurança NOTA Para reduzir os riscos de choques elétricos, queimaduras, ferimentos pessoais ou danos ao equipamento, siga as recomendações básicas de segurança ao usar este aparelho: • Este aparelho não deve ser instalado embutido.
• Evite instalar o aparelho em um local onde exista a incidência de raios solares.
• O aparelho não deve ser instalado atrás de uma porta com fechadura, uma porta de correr ou de uma porta com dobradiças no lado oposto à secadora, de um modo que a completa abertura da porta da lava e seca seja impedida PERIGO • Este aparelho não se destina à utilização por pessoas (inclusive crianças) com capacidades físicas, sensoriais ou mentais reduzidas, ou por pessoas com falta d

In [91]:
instruction, response = generate_instruction_response(
    chunks_300[19],
    hf_pipeline
)

print("Instruction:")
print(instruction)

print()

print("Response:")
print(response)

Instruction:
What should not be done with this appliance?

Response:
Do not install it embedded.


### 6.4 Multiple Example Test

A small sample of chunks is evaluated to assess the quality, consistency, and relevance of the generated instruction-response pairs.


In [93]:
for i in range(92, 100):

    instruction, response = generate_instruction_response(
        chunks_300[i],
        hf_pipeline
    )

    print("=" * 80)
    print(f"CHUNK {i}")

    print()
    print("Instruction:")
    print(instruction)

    print()
    print("Response:")
    print(response)

    print("\n")

CHUNK 92

Instruction:
What installation standard should be followed for this appliance?

Response:
NBR 5410


CHUNK 93

Instruction:
What voltage should be checked before connecting the appliance?

Response:
Check the appliance's voltage before plugging it in.


CHUNK 94

Instruction:
Where should the electrical outlet be placed?

Response:
Near the washing machine for easy access.


CHUNK 95

Instruction:
What safety standard should be followed for this appliance?

Response:
NBR 5410


CHUNK 96

Instruction:
What safety measure ensures protection against electric shocks?

Response:
Grounding prevents electric shocks.


CHUNK 97

Instruction:
What should be done if an electric installation is not done by a qualified professional?

Response:
Hire a qualified electrician.


CHUNK 98

Instruction:
What is the error code for damaged clothes?

Response:
ERROR CODE: 4


CHUNK 99

Instruction:
What should be done with different colored clothes?

Response:
Separate colored clothes for washing

In [94]:
for i in range(55, 61):

    instruction, response = generate_instruction_response(
        chunks_300[i],
        hf_pipeline
    )

    print("=" * 80)
    print(f"CHUNK {i}")

    print()
    print("Instruction:")
    print(instruction)

    print()
    print("Response:")
    print(response)

    print("\n")

CHUNK 55

Instruction:
What should be removed before installing the washing machine?

Response:
Remove the entire packaging, including the bottom base and plastic protectors.


CHUNK 56

Instruction:
What should be checked upon receiving the washing machine?

Response:
All accessories/components were received and in perfect condition.


CHUNK 57

Instruction:
What should be done with plastic bags?

Response:
Throw away plastic bags properly.


CHUNK 58

Instruction:
What should be done to avoid damaging the paint?

Response:
Keep the appliance away from direct sunlight.


CHUNK 59

Instruction:
What should be checked for proper drainage?

Response:
Verify item 3.8 - Installation of Drainage Hose


CHUNK 60

Instruction:
What should be measured for proper installation?

Response:
Comprimento of water supply hose, drain hose, and power cord.




## 👨‍💻 7. Full Dataset Generation

After validating the chunking strategy and the instruction-response generation process, the complete dataset is generated using all relevant chunks extracted from the Midea MFM01D110WB washing machine user manual.

Each relevant chunk is processed by the language model to generate one instruction-response pair. The resulting examples are stored in JSONL format and will later be manually curated to remove invalid or hallucinated samples.

### 7.1 Save Dataset Function

In [105]:
import json

def save_to_jsonl(pairs, output_file):
    """
    Save instruction-response pairs to a JSONL file.
    """

    saved_examples = 0

    with open(output_file, "w", encoding="utf-8") as f:

        for item in pairs:

            instruction = item.get(
                "instruction",
                ""
            ).strip()

            response = item.get(
                "response",
                ""
            ).strip()

            chunk = item.get(
                "chunk",
                ""
            ).strip()

            if not instruction or not response:
                continue

            example = {
                "id": saved_examples,
                "chunk": chunk,
                "instruction": instruction,
                "response": response
            }

            f.write(
                json.dumps(
                    example,
                    ensure_ascii=False
                ) + "\n"
            )

            saved_examples += 1

    print(
        f"Saved {saved_examples} examples to {output_file}"
    )

### 7.2 Generate Dataset

The following step processes all relevant chunks and generates instruction-response pairs using the selected language model.

In [106]:
pairs = []

total_chunks = len(chunks_300)
relevant_chunks = 0
failed_generations = 0

for chunk in tqdm(
    chunks_300,
    desc="Generating instruction-response pairs"
):

    if not is_relevant_chunk(chunk):
        continue

    relevant_chunks += 1

    instruction, response = generate_instruction_response(
        chunk,
        hf_pipeline
    )

    if (
        instruction is not None
        and response is not None
        and instruction.strip().upper() != "NONE"
        and response.strip().upper() != "NONE"
    ):

        pairs.append(
            {
                "chunk": chunk,
                "instruction": instruction.strip(),
                "response": response.strip()
            }
        )

    else:

        failed_generations += 1

Generating instruction-response pairs: 100%|██████████| 302/302 [2:59:43<00:00, 35.71s/it]  


### 7.3 Dataset Statistics

The following statistics summarize the dataset generation process.

In [107]:
print("\nGeneration Summary")
print("-" * 50)
print(f"Total chunks: {total_chunks}")
print(f"Relevant chunks: {relevant_chunks}")
print(f"Generated pairs: {len(pairs)}")
print(f"Failed generations: {failed_generations}")


Generation Summary
--------------------------------------------------
Total chunks: 302
Relevant chunks: 155
Generated pairs: 155
Failed generations: 0


### 7.4 Remove Duplicate Examples

Duplicate instruction-response pairs are removed before saving the final dataset.

In [108]:
unique_pairs = []
seen = set()

for item in pairs:

    key = (
        item["instruction"].strip().lower(),
        item["response"].strip().lower()
    )

    if key not in seen:
        seen.add(key)
        unique_pairs.append(item)

duplicates_removed = len(pairs) - len(unique_pairs)

print(f"Duplicates removed: {duplicates_removed}")
print(f"Unique pairs: {len(unique_pairs)}")

Duplicates removed: 0
Unique pairs: 155


### 7.5 Save Dataset

The final dataset is saved in JSONL format for the fine-tuning stage.

In [109]:
OUTPUT_FILE = "../data/processed/dataset_gerado_raw.jsonl"

save_to_jsonl(
    unique_pairs,
    OUTPUT_FILE
)

print(f"Dataset saved successfully: {OUTPUT_FILE}")

Saved 155 examples to ../data/processed/dataset_gerado_raw.jsonl
Dataset saved successfully: ../data/processed/dataset_gerado_raw.jsonl


### 7.6 Dataset Preview

A random sample of generated examples is displayed for inspection.

In [111]:
import random

sample_size = min(10, len(unique_pairs))

for item in random.sample(
    unique_pairs,
    sample_size
):

    print("-" * 80)

    print("Instruction:")
    print(item["instruction"])

    print()

    print("Response:")
    print(item["response"])

    print()

--------------------------------------------------------------------------------
Instruction:
What helps water reach the fabric better?

Response:
Using soaps increases surface tension on fabrics.

--------------------------------------------------------------------------------
Instruction:
What should be used for cleaning the exterior of a washing machine?

Response:
Use a damp cloth with mild soap solution.

--------------------------------------------------------------------------------
Instruction:
What should be done with the water hose after cleaning?

Response:
Disconnect the hose before cleaning.

--------------------------------------------------------------------------------
Instruction:
What voltage should be checked before connecting the appliance?

Response:
Check the appliance's voltage before plugging it in.

--------------------------------------------------------------------------------
Instruction:
What should be removed before installing the washing machine?

Respons

# 8. Manual Dataset Curation

After automatic generation, all instruction-response pairs were manually reviewed to improve dataset quality and reduce hallucinations.

The review process considered the following criteria:

* Semantic consistency between the instruction and the response.
* Compatibility with the source text chunk.
* Absence of hallucinated information.
* Clarity and objectivity.

Each example received one of the following labels:

* **v** → Valid: the pair is correct and requires no modification.
* **c** → Corrected: the pair contains minor issues and was manually corrected.
* **r** → Remove: the pair contains hallucinations, inconsistencies, or unsupported information and was removed from the dataset.

The curation process aimed to ensure that all final examples were directly supported by the original manual content before being used for fine-tuning.


## 8.1 Initialize Statistics

In [116]:
valid_pairs = []

removed_count = 0
corrected_count = 0
valid_count = 0

total_pairs = len(unique_pairs)

## 8.2 Interactive Curation

In [117]:
for idx, pair in enumerate(unique_pairs):

    print("\n" + "=" * 120)
    print(f"PAIR {idx + 1}/{total_pairs}")
    print("=" * 120)

    print("\nSOURCE CHUNK:\n")
    print(pair["chunk"])

    print("\n" + "-" * 120)

    print("\nINSTRUCTION:\n")
    print(pair["instruction"])

    print("\nRESPONSE:\n")
    print(pair["response"])

    while True:
        decision = input(
            "\n[v] Valid | [c] Correct | [r] Remove: "
        ).strip().lower()

        if decision in {"v", "c", "r"}:
            break

        print("Invalid option. Please enter v, c, or r.")

    if decision == "v":

        valid_pairs.append({
            "chunk": pair["chunk"],
            "instruction": pair["instruction"],
            "response": pair["response"]
        })

        valid_count += 1

    elif decision == "c":

        print("\nCorrected instruction:")
        new_instruction = input().strip()

        print("\nCorrected response:")
        new_response = input().strip()

        valid_pairs.append({
            "chunk": pair["chunk"],
            "instruction": new_instruction,
            "response": new_response
        })

        corrected_count += 1

    elif decision == "r":

        removed_count += 1


PAIR 1/155

SOURCE CHUNK:

Este manual foi feito especialmente para que você conheça todas as características do seu aparelho, além de informações sobre manutenção, execução de serviços e claro, como obter o máximo das suas funcionalidades.

------------------------------------------------------------------------------------------------------------------------

INSTRUCTION:

What is the purpose of this washing machine manual?

RESPONSE:

To provide information on features, maintenance, service execution, and maximizing functionality.

PAIR 2/155

SOURCE CHUNK:

1.1 - Medidas Importantes de Segurança NOTA Para reduzir os riscos de choques elétricos, queimaduras, ferimentos pessoais ou danos ao equipamento, siga as recomendações básicas de segurança ao usar este aparelho: • Este aparelho não deve ser instalado embutido.

------------------------------------------------------------------------------------------------------------------------

INSTRUCTION:

What should not be done with thi

## 8.3 Final Statistics

In [118]:
final_pairs = len(valid_pairs)

removed_percentage = (
    removed_count / total_pairs
) * 100

corrected_percentage = (
    corrected_count / total_pairs
) * 100

accepted_percentage = (
    final_pairs / total_pairs
) * 100

# Consistency check
assert (
    final_pairs + removed_count == total_pairs
), "Statistics are inconsistent."

print("\n" + "=" * 60)
print(f"Total generated pairs: {total_pairs}")
print(f"Accepted without changes: {valid_count}")
print(f"Corrected pairs: {corrected_count}")
print(f"Final accepted pairs: {final_pairs}")
print(f"Removed pairs: {removed_count}")
print(f"Accepted percentage: {accepted_percentage:.2f}%")
print(f"Corrected percentage: {corrected_percentage:.2f}%")
print(f"Removed percentage: {removed_percentage:.2f}%")
print("=" * 60)


Total generated pairs: 155
Accepted without changes: 45
Corrected pairs: 80
Final accepted pairs: 125
Removed pairs: 30
Accepted percentage: 80.65%
Corrected percentage: 51.61%
Removed percentage: 19.35%


In [124]:
print(valid_pairs[0])

{'chunk': 'Este manual foi feito especialmente para que você conheça todas as características do seu aparelho, além de informações sobre manutenção, execução de serviços e claro, como obter o máximo das suas funcionalidades.', 'instruction': 'What is the purpose of this washing machine manual?', 'response': 'To provide information on features, maintenance, service execution, and maximizing functionality.'}


## 8.4 Save Final Dataset

In [136]:
import json

with open("../data/processed/dataset_gerado_not_cleaned.jsonl", "w", encoding="utf-8") as f:

    for item in valid_pairs:

        example = {
            "instruction": item["instruction"],
            "response": item["response"]
        }

        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print(f"\nDataset saved with {len(valid_pairs)} valid examples.")


Dataset saved with 125 valid examples.


## 8. 5 Cleaning Final Dataset (Curation Mistake)

In [137]:
import json

cleaned = []

with open("../data/processed/dataset_gerado_not_cleaned.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)

        if item.get("instruction") in ["", "c", None]:
            continue
        if item.get("response") in ["", None]:
            continue

        cleaned.append(item)

with open("../data/processed/dataset_gerado.jsonl", "w", encoding="utf-8") as f:
    for item in cleaned:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Clean dataset saved.")
print("Total valid pairs:", len(cleaned))

Clean dataset saved.
Total valid pairs: 124


## 8.5 Save Curation Report

In [138]:
total_pairs = 155
auto_removed = 30
manual_removed = 1

cleaned_pairs = len(cleaned) 

assert cleaned_pairs == total_pairs - auto_removed - manual_removed

corrected_pairs = 80
valid_without_change = cleaned_pairs - corrected_pairs 

curation_report = {
    "total_generated_pairs": total_pairs,
    "auto_removed_pairs": auto_removed,
    "manual_removed_pairs": manual_removed,
    "cleaned_pairs": cleaned_pairs,
    "corrected_pairs": corrected_pairs,
    "valid_without_change": valid_without_change,
    "removed_total": auto_removed + manual_removed,
    "removed_percentage": round(((auto_removed + manual_removed) / total_pairs) * 100, 2),
    "corrected_percentage": round((corrected_pairs / cleaned_pairs) * 100, 2)
}

with open("../reports/curation_report.json", "w", encoding="utf-8") as f:
    json.dump(curation_report, f, indent=4, ensure_ascii=False)

print(json.dumps(curation_report, indent=4, ensure_ascii=False))

{
    "total_generated_pairs": 155,
    "auto_removed_pairs": 30,
    "manual_removed_pairs": 1,
    "cleaned_pairs": 124,
    "corrected_pairs": 80,
    "valid_without_change": 44,
    "removed_total": 31,
    "removed_percentage": 20.0,
    "corrected_percentage": 64.52
}
